In [ ]:
import astropy.units as u
from ao_tools import etc

In [ ]:
verbose = False

In [ ]:
options = etc.ETCOptions(
    instrument = 'GIRMOS',                               # use 'GIRMOS-KMOS' to use KMOS spatial resolution
)
options.config_fov(
    fov = 2.0 * u.arcsec,                                # Valid Choices: 1.0, 2.0, 4.0 arcsec
    pa = 0.0 * u.deg                                     # Field rotation (position angle) measured East from North
)
options.config_spec(
    band = 'HK',                                         # Valid Choices: R~3000: 'YJ', 'JH', 'HK'. R~8000: 'J', 'H', 'K'
    R = 3000,                                            # Spectral Resolution: R~3000 or R~8000, must be consistent with band choice
    force_spec_range = None                              # Optionally provide range for spectrum plots
)
options.config_target(
    #-----------------------
    #--- Target Location ---
    #-----------------------

    # ra = 0.0*u.deg, dec = 0.0*u.deg,                   # RA/Dec are only needed if you specify an asterism by id in config_ao()
    r = 0.0*u.arcsec, theta = 0.0*u.deg,                 # Target's polar coordinates within FOR relative to the LGS which are at r=30", theta=45°, 135°, 225°, 315°
    # r = 30.0*u.arcsec, theta = 45.0*u.deg,             # Location of one of the LGS
    redshift = 2.0,                                      # Redshift is needed if you only specify a rest wavelength or set flux using lsfr/Av

    #-----------------------
    #--- Spatial Profile ---
    #-----------------------

    # ** Option 1: Point Source **
    spatial_type       = etc.SpatialType.POINT,

    # ** Option 2: Gaussian Profile **
    # spatial_type     = etc.SpatialType.GAUSSIAN,
    # size             = 1.0*u.arcsec,                  # Gaussian FWHM (arcsec)

    # ** Option 3: Sersic Profile **
    # spatial_type     = etc.SpatialType.SERSIC,
    # sersic_Re        = 0.75*u.kpc,                    # Sersic effective radius (arcsec or kpc)
    # sersic_n         = 1.0,                           # Sersic index
    # sersic_q         = 0.5,                           # Sersic axis ratio
    # sersic_pa        = 45.0*u.deg,                    # Sersic position angle

    # ** Option 4: From File **
    # spatial_type     = etc.SpatialType.FILE,
    # spatial_filename = 'path/to/profile/file.fits',

    #------------------------
    #--- Spectral Profile ---
    #------------------------

    # ** Option 1: Gaussian Emission Line **
    spectral_type    = etc.SpectralType.GAUSSIAN,
    wvl_rest         = 0.656281 * u.micron,             # Rest wavelength of emission line in air
    # wvl              = 1.968843 * u.micron,           # Observed wavelength of emission line in air (optional if wvl_rest and redshift provided)
    flux             = 1e-16*u.erg/u.s/u.cm**2,         # Either integrated flux (eg. erg/s/cm**2) or surface brightness (eg. erg/s/cm**2/arcsec**2)
    # lsfr = 1.5, Av   = 1.0*u.mag, Av_MW = 0.0*u.mag,  # Can be used instead of specifying flux directly
    dispersion       = 130*u.km/u.s,
    continuum        = 1e-17*u.erg/u.s/u.cm**2/u.angstrom,
    # equivalent_width = 50*u.Angstrom,                 # Can be used instead of specifying continuum directly

    # ** Option 2: Uniform **
    # spectral_type    = etc.SpectralType.UNIFORM,      # NOTE this skips convolution with the PSF so that the flux remains uniform in all pixels

    # ** Option 3: From File **
    # spectral_type    = etc.SpectralType.FILE,
    # spectrum_filename = 'path/to/profile/file.csv',   # File either has one column with flux or two columns with wvl then flux
    # spectrum_units   = u.erg/u.s/u.cm**2/u.arcsec**2/u.angstrom, # Either surface brightness (eg. erg/s/cm**2/arcsec**2/A) or integrated flux (eg. erg/s/cm**2/A)

    #------------------------
    #--- Velocity Profile ---
    #------------------------

    # ** Option 1: No Velocity Profile **
    velocity_type    = etc.VelocityType.NONE,

    # ** Option 2: Rotating Disk Profile (requires spatial_type=SpatialType.SERSIC) **
    # velocity_type    = etc.VelocityType.ROTATING_DISC,
    # sersic_disc_inc  = 15.0*u.deg,                    # Disc inclination (0=face-on, 90=edge-on)
    # sersic_disc_vmax = 200*u.km/u.s,                  # Disc maximum rotation velocity
    # sersic_disc_rturn = 0.5*u.arcsec,                 # Disc turnover radius (arcsec or kpc)

    # ** Option 3: From File **
    # velocity_type     = etc.VelocityType.FILE,
    # velocity_filename = 'path/to/profile/file.fits',  # 2D at same resolution as spatial profile or 40x40 matching IFU
    # velocity_units    = u.km/u.s,

    #-------------------------------
    #--- Additional Point Source ---
    #-------------------------------

    # points=[
    #    {'x': -0.5*u.arcsec, 'y':  0.5*u.arcsec, 'flux':1e-17*u.erg/u.s/u.cm**2, 'dispersion': 100*u.km/u.s, 'vel_offset': 50*u.km/u.s},
    #    {'x':  0.5*u.arcsec, 'y':  0.5*u.arcsec, 'flux':1e-17*u.erg/u.s/u.cm**2, 'dispersion': 100*u.km/u.s, 'vel_offset': 50*u.km/u.s},
    #    {'x': -0.5*u.arcsec, 'y':  0.5*u.arcsec, 'spectrum_filename': 'path/to/profile/file.fits', 'spectrum_units': u.erg/u.s/u.cm**2/u.arcsec**2/u.angstrom},
    # ]
)
options.config_ao(
    psf_type         = etc.PSFType.MOAO,                # etc.PSFType.MOAO or etc.PSFType.SEEING (seeing limited PSF)
    # wvl              = 2.0 * u.micron,                # Optional wavelength used for PSF simulation, target wvl used if not provided

    # ** Option 1: Provide polar coordinates and magnitudes of NGS within FOR relative to LGS (r in arcsec, theta in deg) **
    ngs = [
        {"r": 30.0, "theta":   0.0, "mag": 17.0},
        {"r": 30.0, "theta": 120.0, "mag": 17.0},
        {"r": 30.0, "theta": 240.0, "mag": 17.0},
    ],

    # ** Option 2: Reference asterism in GNAO asterism catalog **
    # asterism_id     = 884058320,
    # mock_asterism   = True,                           # True indicates asterism is not in the vicinity of target
)
options.config_atm(
    atm               = 'median',                       # Options: 25p, median, 75p
    zenith_angle      = 20*u.deg,                       # Options: 0-40 deg
    water_vapor       = 1.6*u.mm,                       # Options: 1.0, 1.6, 3.0, 5.0 mm
)
options.config_analysis(
    T_exp             = 600*u.s,                        # Time per exposure
    N_exp             = 24,                             # Number of on-source exposures
    # aperture_diameter = [2,2,2],                      # Specify a pixel aperture over which to compute S/N [dy,dx,dwvl] NOTE: dwvl is relative to the target wavelength
    # aperture_spaxel0  = [19,19],                      # Specify top left spaxel of the aperture [y,x] NOTE: spaxels are zero indexed
    # aperture_wvl0     = 2.0 * u.micron,               # Specify the starting wavelength for the aperture NOTE: centered on Gaussian line wavelength or PSF wavelength if not specified
)
options.config_finish()

if verbose:
    options.print()

In [ ]:
psf_data = etc.get_psf(options, verbose=verbose)
# etc.save_psf(psf_data, 'psf.fits')

In [ ]:
result = etc.compute(options, psf_data, verbose=verbose)
etc.save_fits_signal(options, result, 'signal.fits')

In [ ]:
image_file = None
# image_file = '/path/to/image.fits' # Optional: provide a FITS image with valid WCS for the FOR plot

# Show all plots
etc.plot_all(options, psf_data, result, image_file=image_file)

# Show individual plots
# etc.plot_psf(psf_data)
# etc.plot_for(options, image_file=image_file)
# etc.plot_target_model(options, result)
# etc.plot_radiance_comparison(options, result)
# etc.plot_peak_signal(options, result, include_noise=True)
# etc.plot_peak_spectrum_snr(options, result)
# etc.plot_peak_spatial_snr(options, result)

if verbose:
    etc.print_peak_signal(options, result)

In [ ]:
if options.aperture_mask is not None:
    if verbose:
        etc.print_max_radiance_in_aperture(options, result)

    aperture_result = etc.compute_snr_in_aperture(result, options.aperture_mask)
    etc.print_aperture_SNR(options, aperture_result)